In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import matplotlib as mpl
import textwrap

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
# little function to define the file root on different machines
def find_f_root(start_path: Path = Path.cwd(), anchor: str = "CASA0004_work") -> Path:
    """
    Traverse up from the start_path until the anchor folder is found. Returns the path to the anchor folder.
    """
    for parent in [start_path] + list(start_path.parents):
        if parent.name == anchor:
            return parent
    raise FileNotFoundError(f"Anchor folder '{anchor}' not found in path hierarchy.")
  
f_root = find_f_root()

In [ ]:
# Load nvf lookup
victim_lookup = pd.read_csv(
    f_root / "data/csew/exp_for_inla_stage/perc_pop_victim.csv")
victim_lookup

# Visualisation

In [ ]:
small_geogr = gpd.read_file(f_root / "data/geographies/ltla/export_small_geogr_ew.shp") #lower-tier local authorities, BSC | district/unitary
small_geogr.info()

In [ ]:
gor_gdf = (
    small_geogr[["gor", "geometry"]]
    .dissolve(by="gor")
    .reset_index()
)

small_geogr_merged = gor_gdf.merge(
    victim_lookup,
    on="gor",
    how="left",
    validate="one_to_many"
)

In [ ]:
gor_gdf.info()

In [ ]:
small_geogr_merged.info()

In [ ]:
# pip install adjustText

In [ ]:
import matplotlib.patheffects as pe
# from adjustText import adjust_text

In [ ]:
# ----------------------------
# choose what to plot
# ----------------------------
plot_col  = "percent_pop_victim"   # change this to any column in victim_lookup
label_fmt = "{:.1f}%"              # label format shown on each GOR

# optional ordering
age_order = ["age_16_24", "age_25_34", "age_35_64", "age_65_plus"]
emp_order = sorted(victim_lookup["remploya"].dropna().unique())

# ----------------------------
# build regional plotting gdf
# ----------------------------
gor_shapes = (
    small_geogr_merged[["gor", "geometry"]]
    .dissolve(by="gor")
    .reset_index()
)

plot_gdf = gpd.GeoDataFrame(
    gor_shapes.merge(
        victim_lookup[["gor", "agelong", "remploya", plot_col]],
        on="gor",
        how="left",
        validate="one_to_many"
    ),
    geometry="geometry",
    crs=small_geogr.crs
)

# keep only categories present
age_order = [x for x in age_order if x in plot_gdf["agelong"].unique()]
emp_order = [x for x in emp_order if x in plot_gdf["remploya"].unique()]

# common colour scale across all panels
vmin, vmax = plot_gdf[plot_col].min(), plot_gdf[plot_col].max()
cmap, norm = "Reds", mpl.colors.Normalize(vmin=vmin, vmax=vmax)

In [ ]:
plt.rcParams["font.family"] = "Roboto Slab"

age_labels = {"age_16_24":"16–24","age_25_34":"25–34","age_35_64":"35–64","age_65_plus":"65+"}
emp_labels = {"employed":"Employed","unemployed_or_economically_inactive":"Unemployed or\neconomically inactive"}
label_offsets = {"London": (30000, 0)}

fig, axes = plt.subplots(len(age_order),len(emp_order),figsize=(6,12),
                         gridspec_kw={"wspace":-.35,"hspace":-.08})
axes = np.atleast_2d(axes)

for i, age in enumerate(age_order):
    for j, emp in enumerate(emp_order):
        ax = axes[i,j]
        sub = plot_gdf.query("agelong == @age and remploya == @emp").copy()

        sub.plot(column=plot_col,cmap=cmap,norm=norm,ax=ax,
                 edgecolor="white",linewidth=.25,
                 missing_kwds={"color":"lightgrey"})

        pts = sub.representative_point()
        for gor,x,y,v in zip(sub["gor"],pts.x,pts.y,sub[plot_col]):
            if pd.notna(v):
                dx,dy = label_offsets.get(gor,(0,0))
                ax.text(x+dx,y+dy,label_fmt.format(v),ha="center",va="center",fontsize=10,
                        path_effects=[pe.Stroke(linewidth=3,foreground="white"),pe.Normal()])

        if i == 0: ax.set_title(emp_labels.get(emp,emp),fontsize=11,pad=1)
        if j == 0:
            ax.annotate(age_labels.get(age,age),xy=(0,.5),xytext=(-2,0),
                        xycoords="axes fraction",textcoords="offset points",
                        ha="right",va="center",rotation=90,fontsize=11)
        ax.set_axis_off()

fig.subplots_adjust(left=.01,right=.89,top=.89,bottom=.01,wspace=-.75,hspace=-.38)

sm = mpl.cm.ScalarMappable(norm=norm,cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm,ax=axes,shrink=.35,pad=.02)
cbar.set_label("% of sample who experienced VAWG")
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=8)

# fig.suptitle(
#     "% of female population aged over 16\nestimated to be VAWG survivors,\n"
#     "by age group and economic activity status",
#     fontsize=14
# )

plt.show()

In [ ]:
fig.savefig(
    f_root / "figures/vawg_prevalence_by_category_final.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

 Plotting width of confidence intervals

In [ ]:
plot_col = "perc_ci_upper_lower"

lookup_plot = victim_lookup.assign(
    **{plot_col:
       pd.to_numeric(victim_lookup["percent_ci_upper"], errors="coerce")
       - pd.to_numeric(victim_lookup["percent_ci_lower"], errors="coerce")}
)

plot_gdf = gpd.GeoDataFrame(
    gor_shapes.merge(
        lookup_plot[["gor", "agelong", "remploya", plot_col]],
        on="gor",
        how="left",
        validate="one_to_many"
    ),
    geometry="geometry",
    crs=gor_shapes.crs
)

vmin, vmax = plot_gdf[plot_col].min(), plot_gdf[plot_col].max()
cmap, norm = "Blues", mpl.colors.Normalize(vmin=vmin, vmax=vmax)

In [ ]:
plt.rcParams["font.family"] = "Roboto Slab"

age_labels = {"age_16_24":"16–24","age_25_34":"25–34","age_35_64":"35–64","age_65_plus":"65+"}
emp_labels = {"employed":"Employed","unemployed_or_economically_inactive":"Unemployed or\neconomically inactive"}
label_offsets = {"London": (30000, 0)}

fig, axes = plt.subplots(len(age_order),len(emp_order),figsize=(6,12),
                         gridspec_kw={"wspace":-.35,"hspace":-.08})
axes = np.atleast_2d(axes)

for i, age in enumerate(age_order):
    for j, emp in enumerate(emp_order):
        ax = axes[i,j]
        sub = plot_gdf.query("agelong == @age and remploya == @emp").copy()

        sub.plot(column=plot_col,cmap=cmap,norm=norm,ax=ax,
                 edgecolor="white",linewidth=.25,
                 missing_kwds={"color":"lightgrey"})

        pts = sub.representative_point()
        for gor,x,y,v in zip(sub["gor"],pts.x,pts.y,sub[plot_col]):
            if pd.notna(v):
                dx,dy = label_offsets.get(gor,(0,0))
                ax.text(x+dx,y+dy,label_fmt.format(v),ha="center",va="center",fontsize=10,
                        path_effects=[pe.Stroke(linewidth=3,foreground="white"),pe.Normal()])

        if i == 0: ax.set_title(emp_labels.get(emp,emp),fontsize=11,pad=1)
        if j == 0:
            ax.annotate(age_labels.get(age,age),xy=(0,.5),xytext=(-2,0),
                        xycoords="axes fraction",textcoords="offset points",
                        ha="right",va="center",rotation=90,fontsize=11)
        ax.set_axis_off()

fig.subplots_adjust(left=.01,right=.89,top=.89,bottom=.01,wspace=-.75,hspace=-.38)

sm = mpl.cm.ScalarMappable(norm=norm,cmap=cmap)
sm.set_array([])
cbar = fig.colorbar(sm,ax=axes,shrink=.35,pad=.02)
cbar.set_label("95% confidence interval width (percentage points)")
cbar.outline.set_visible(False)
cbar.ax.tick_params(labelsize=8)

# fig.suptitle(
#     "Width of confidence intervals (%)\nobtained through bootstrapping",
#     fontsize=14
# )

plt.show()

In [ ]:
fig.savefig(
    f_root / "figures/vawg_prevalence_confidence_intervals_by_category_final.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)